# Controlled synthetic stress markets — Student-t, stationary AR, and whipsaw

This extends the existing financial factor simulator, not the neural models. Three sources isolate different
mechanisms: **shared-scale Student-t log innovations** (`nu=8`), **stationary Gaussian AR(1) log returns**
(`phi=0.4`), and **switching Gaussian reversal dynamics** (`phi=[0.1,-0.65]`). No added jumps.

Each config retains 12 assets, 8,000 business weekdays, lookback 60, horizon 1, and the same Vanilla
architecture/optimizer as 4A/4B. Calendar: 1990-01-01–2020-08-28; cutoffs 2009-02-27 and 2014-11-28.
Usable windows: **4,940 train / 1,500 validation / 1,500 held out**. RF subtraction is disabled.
Only **training dates** enter diagnostics. Counting held-out availability is not performance evaluation.

These experiments validate the specified mechanisms, not empirical realism or forecasting superiority.
All theoretical moments/ACFs below concern **daily log returns**; observable pipeline returns remain
arithmetic decimals `expm1(x)`. Student-t log variance/fourth moments are finite for our `nu>4`, but
**exponentiated Student-t arithmetic returns do not have finite positive population moments**.
Do not interpret finite sample arithmetic moments as evidence otherwise. No tail clipping or path screening.

## Setup

Enable Internet in Kaggle. CPU is sufficient; no long neural training runs are included.
The established sync cell selects `feature/synthetic-market`. Change that branch only after merging.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/synthetic-market"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev]'
assert _exit_code == 0, "Dependency installation failed."

## Mathematical and backward-compatibility tests

Tests include pre-extension numerical hashes for all three original scenarios, common-scale Student-t
construction, sampling-aware moment checks, causal AR dynamics, within-spell reversals, and pipeline isolation.
Log returns and all other stable arrays always require exact hashes. Returns and prices require exact hashes
under the reference NumPy 2.4.0; other versions check their mathematical reconstruction with tight float64
tolerances. Bitwise verification of these two derived arrays requires the reference NumPy version.
Fixtures are never regenerated automatically. No real model training, feature extraction or forecast evaluation is launched.

In [ ]:
!pytest -q tests/unit/test_synthetic_market_stage4c.py tests/unit/test_synthetic_market.py \
    tests/unit/test_synthetic_market_pipeline.py tests/semantic/test_data_leakage.py
assert _exit_code == 0, "Synthetic stress regression tests failed."

## Mathematical specification

Write $v_t=\sigma_{z_t}(B_{z_t}f_t+D_{z_t}\epsilon_t)$, with independent standard Gaussian shocks.

- **Student-t:** $u_t=v_t\sqrt{(\nu-2)/W_t}$ with one independent $W_t\sim\chi^2_\nu$ shared across the
  entire vector each day. Fixed regime, no AR or jumps. Covariance stays $\Sigma$; marginal excess kurtosis
  is $6/(\nu-4)=1.5$ at `nu=8`. A fifth isolated RNG stream leaves the original four streams untouched.
- **Mean-reverting returns:** $y_0=v_0$ and $y_t=\phi y_{t-1}+\sqrt{1-\phi^2}v_t$, $x_t=\mu+y_t$.
  This Gaussian initialization is stationary: covariance $\Sigma$, lag-$k$ covariance $\phi^k\Sigma$.
  `abs(phi)<1` is required. This describes return deviations, **not OU/mean-reverting price levels**.
- **Whipsaw:** $y_t=\phi_{z_t}y_{t-1}+\sqrt{1-\phi_{z_t}^2}v_t$, $x_t=\mu_{z_t}+y_t$ with persistent
  Markov states. Normal/whipsaw coefficients are `0.1/-0.65`; driver volatility/exposure parameters retain
  the 4B normal/stress settings. Initial deviation is Gaussian under the initial state, **not a draw from
  the full switching stationary law**. Driver covariance and shock covariance are reported separately;
  resulting regime-conditional return covariance and stationary ACF are **not claimed analytically**.

Prices compound across all states without resets. Regime labels, Student-t scales and AR coefficients are
separate ground truth; they never enter the forecasting ReturnTable. Unvalidated mechanism combinations fail.

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from diffusion_portfolio.config import load_config
from diffusion_portfolio.data.sources.synthetic_market import simulate_market
from diffusion_portfolio.evaluation.synthetic_market import market_diagnostics, plot_market_diagnostics, plot_stress_diagnostics

SCENARIOS = ("synthetic_student_t", "synthetic_mean_reverting", "synthetic_whipsaw")
configs = {name: load_config(f"configs/{name}.yaml") for name in SCENARIOS}
markets = {name: simulate_market(name, **cfg.data.source_options) for name, cfg in configs.items()}
OUT = Path("/kaggle/working/synthetic_market_stage4c_validation")
OUT.mkdir(exist_ok=True)
for name, result in markets.items():
    print(name, result.returns.shape, result.dates[0].date(), "to", result.dates[-1].date())
    directory = OUT / name
    directory.mkdir(exist_ok=True)
    (directory / "parameters.json").write_text(json.dumps(result.parameters, indent=2, allow_nan=False))
    shutil.copy2(f"configs/{name}.yaml", directory / "config.yaml")

## Training-period moments and temporal dependence

Lag pairs must lie inside the same **uninterrupted regime spell**, including intermediate observations.
Switching AR coefficients describe conditional slopes, not exact stationary correlations. Blank configured
return covariance/volatility for whipsaw is deliberate. Raw sign reversals and reversals about the known log mean are reported separately.

In [ ]:
reports = {}
for name, cfg in configs.items():
    report = market_diagnostics(markets[name], end_date=cfg.data.train_end)
    reports[name] = report
    print(name, report["start"], "to", report["end"], report["covariance_interpretation"])
    columns = ["regime", "observations", "occupancy", "configured_mean_log_volatility", "observed_mean_log_volatility",
               "configured_mean_correlation", "observed_mean_correlation", "observed_duration_days"]
    display(pd.DataFrame(report["summary"])[columns].set_index("regime"))
    display(pd.DataFrame(report["dynamics"])[["state", "lag", "pairs", "configured_lag_slope", "theoretical_correlation",
                                           "observed_mean_correlation", "raw_sign_reversal_rate", "centered_sign_reversal_rate"]])
    (OUT / name / "training_diagnostics.json").write_text(json.dumps(report, indent=2, allow_nan=False))

## Student-t tails and cross-asset co-exceedances

Quantiles and exceedance thresholds are standardized by **configured log-return variance**, not a fitted
threshold. Joint statistics are finite-threshold lower-tail co-exceedances, not asymptotic tail-dependence
estimates. Sample kurtosis can be unstable: default `nu=8` has a finite fourth but no eighth moment.
The kurtosis-SE unit test therefore uses `nu=12`, while default-8 covariance and tail tests remain separate.

In [ ]:
tails = reports["synthetic_student_t"]["tails"]
labels = markets["synthetic_student_t"].columns
display(pd.DataFrame({"Observed log excess kurtosis": tails["observed_excess_kurtosis"],
                      "Configured": tails["configured_excess_kurtosis"]}, index=labels))
quantiles = pd.DataFrame(tails["observed_quantiles_by_asset"], index=tails["quantile_levels"], columns=labels)
quantiles["Configured standardized quantile"] = tails["configured_quantiles"]
display(quantiles)
display(pd.DataFrame([{"Threshold (log std)": row["threshold_log_standard_deviations"],
                      "Observed two-sided mean": np.mean(row["observed_two_sided_by_asset"]),
                      "Configured Student-t": row["configured_two_sided_probability"],
                      "Matched Gaussian": row["gaussian_two_sided_probability"],
                      "Observed lower pair co-exceedance": row["observed_mean_lower_pair_probability"]}
                     for row in tails["exceedances"]]))

## Stress-event views

Each figure uses training data only. The whipsaw trajectory starts near the **first** observed whipsaw state,
not a hand-picked successful reversal. Its whole-sample regime occupancy/transitions appear below.

In [ ]:
for name, cfg in configs.items():
    fig = plot_stress_diagnostics(markets[name], end_date=cfg.data.train_end)
    plt.show()
    fig.savefig(OUT / name / "training_stress_diagnostics.png", dpi=120, bbox_inches="tight")

In [ ]:
report = reports["synthetic_whipsaw"]
print("Configured and observed normal/whipsaw transition probabilities:")
display(pd.DataFrame(report["configured_transition"], index=["normal", "whipsaw"], columns=["normal", "whipsaw"]))
display(pd.DataFrame(report["observed_transition"], index=["normal", "whipsaw"], columns=["normal", "whipsaw"]))
print("Gaussian-driver covariance and actual AR-shock covariance are distinct; return covariance is simulation-based.")

## The same ReturnTable, normalization and causal windows as real datasets

The adapter returns **only dates, arithmetic returns and asset labels**. Train-only normalization is fitted
on the first 5,000 observations, then saved/restored. Counts below include held-out availability, not test-set
statistics or performance. Ground-truth labels are never concatenated with model inputs.

In [ ]:
from diffusion_portfolio.data.registry import load_return_table
from diffusion_portfolio.data.preparation import prepare_return_data, save_standardizer

rows = []
for name, cfg in configs.items():
    table = load_return_table(name, **cfg.data.source_options)
    assert set(vars(table)) == {"dates", "returns", "columns"}
    np.testing.assert_array_equal(table.returns, markets[name].returns)
    datasets, scaler = prepare_return_data(cfg)
    train_only = table.returns[table.dates <= pd.Timestamp(cfg.data.train_end)]
    np.testing.assert_array_equal(scaler.mean, train_only.mean(0))
    save_standardizer(OUT / name / "standardizer.npz", scaler)
    restored, _ = prepare_return_data(cfg, standardizer_path=OUT / name / "standardizer.npz")
    np.testing.assert_array_equal(restored.val.model_windows.history, datasets.val.model_windows.history)
    rows.append({"Dataset": name, "Assets": len(table.columns), "Train windows": len(datasets.train),
                 "Validation windows": len(datasets.val), "Held-out windows": len(datasets.test),
                 "History shape": tuple(datasets.train[0]["history"].shape),
                 "Target shape": tuple(datasets.train[0]["target"].shape)})
    del datasets, restored
display(pd.DataFrame(rows).set_index("Dataset"))

## Small CPU model smoke tests

The existing tool runs forward/loss/backward on **two training examples** only, with no optimizer update,
scenario sampling, checkpoint writes, or test-period evaluation. These losses are checks, not research results.

In [ ]:
!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/data_smoke_test.py \
    --config configs/synthetic_student_t.yaml --model-smoke --smoke-batch-size 2 --device cpu
assert _exit_code == 0, "synthetic_student_t model smoke failed."

In [ ]:
!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/data_smoke_test.py \
    --config configs/synthetic_mean_reverting.yaml --model-smoke --smoke-batch-size 2 --device cpu
assert _exit_code == 0, "synthetic_mean_reverting model smoke failed."

In [ ]:
!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/data_smoke_test.py \
    --config configs/synthetic_whipsaw.yaml --model-smoke --smoke-batch-size 2 --device cpu
assert _exit_code == 0, "synthetic_whipsaw model smoke failed."

## Optional manual dataset export

After generation, select this cell to save `observations.npz` (dates, asset labels, arithmetic returns,
and prices when available) and a separate `ground_truth.npz` (aligned dates and hidden diagnostic arrays).
The checked-out full commit SHA is added to `parameters.json`. Files preserve the complete simulated
trajectory, including held-out observations; this is an export, not a test-period evaluation.
Forecasting continues through the existing returns-only source adapter, never the ground-truth archive.

In [ ]:
revision = !git rev-parse HEAD
assert len(revision) == 1, "Could not read the checked-out source revision."
source_revision = revision[0].strip()
assert len(source_revision) == 40 and all(c in "0123456789abcdef" for c in source_revision), "Invalid source revision; export stopped."

for name, result in markets.items():
    directory = OUT / name
    observations = {"dates": result.dates.values, "columns": np.asarray(result.columns), "returns": result.returns}
    if result.prices is not None:
        observations["prices"] = result.prices
    np.savez_compressed(directory / "observations.npz", **observations)
    np.savez_compressed(directory / "ground_truth.npz", dates=result.dates.values, **result.ground_truth)
    parameters = {**result.parameters, "source_revision": source_revision}
    (directory / "parameters.json").write_text(json.dumps(parameters, indent=2, allow_nan=False))
    print("Exported observations and separate ground truth:", directory)

## Saved records and interpretation

Outputs are isolated under `/kaggle/working/synthetic_market_stage4c_validation/<scenario>/`.
Keep configuration, resolved parameters/NumPy version, scaler and training-only report/figures together.
Optional export records the full checkout SHA alongside parameters, with observations and ground truth in
separate archives. The source adapter continues to expose only arithmetic observations.

Implementation agreement with mathematical specifications and the presence of stress mechanisms do not
establish empirical financial realism or predictive benefits. Heavy-tailed arithmetic population moments,
switching AR stationary covariance and cross-regime ACF must not be inferred from invalid fixed-regime formulas.
The simulator retains a weekday calendar and floating-point range checks; extreme unrepresentable paths fail
rather than being clipped. No alpha-stable, intraday/Hawkes, synthetic geometry, or new neural architecture is
implemented. The original 4A/4B notebook and historical real-data experiments remain separate.